# explore_dbt -- dbt model development against local DuckDB

**Flow**
1. Run models locally against the Parquet cache -- fast, free, no cluster
2. Query results via DuckDB to validate logic
3. Compile to inspect generated SQL for both DuckDB and Spark targets
4. Push to Databricks when satisfied

Prereq: run the Sync cell in `explore_sql.ipynb` first to populate `.cache/`.

## Sanity checks

In [ ]:
# Confirm dbt can see the project
!cd /workspaces/your-project/local-devex/dbt && dbt --version && dbt ls --target local

## Run all models locally

Reads from `.cache/*.parquet`, writes to `.cache/lakehouse_dev.duckdb`. Typically completes in ~3 seconds.

In [ ]:
!cd /workspaces/your-project/local-devex/dbt && dbt run --target local

## Run a single model (or model + downstream)

Use `--select model_name` for a single model, `model_name+` to include everything downstream.

In [ ]:
!cd /workspaces/your-project/local-devex/dbt && dbt run --target local --select events_enriched

In [ ]:
# Run a model and everything downstream of it
!cd /workspaces/your-project/local-devex/dbt && dbt run --target local --select events_enriched+

## Query results

After `dbt run`, silver and gold tables are available in `.cache/lakehouse_dev.duckdb`.

In [ ]:
import os
import duckdb

DB_PATH = os.environ.get("LAKEHOUSE_DB_PATH", "/workspaces/your-project/local-devex/.cache/lakehouse_dev.duckdb")
conn = duckdb.connect(DB_PATH)

# List all available tables and views
conn.sql("SHOW ALL TABLES").df()

In [ ]:
conn.sql("SELECT * FROM silver.events_enriched LIMIT 20").df()

In [ ]:
conn.sql("SELECT * FROM gold.daily_metrics ORDER BY event_date DESC LIMIT 20").df()

## Compile -- inspect generated SQL

Use this to verify what SQL dbt actually generates for each target before running.
Especially useful for checking that the `cross_db` macros resolve correctly.

In [ ]:
# Compile for local (DuckDB) -- shows json_extract_string, count(*) filter, etc.
!cd /workspaces/your-project/local-devex/dbt && dbt compile --target local --select events_enriched
!cat /workspaces/your-project/local-devex/dbt/target/compiled/my_lakehouse/models/silver/events_enriched.sql

In [ ]:
# Compile for prod (Spark) -- shows get_json_object, count(case when ...), etc.
# Does not require a live Databricks connection.
!cd /workspaces/your-project/local-devex/dbt && dbt compile --target prod --select events_enriched
!cat /workspaces/your-project/local-devex/dbt/target/compiled/my_lakehouse/models/silver/events_enriched.sql

## Push to Databricks

When you are satisfied with local results, push to the prod target.
This writes to Unity Catalog silver schema on your Databricks workspace.

In [ ]:
import os
import subprocess

host = os.environ["DATABRICKS_HOST"]
token = subprocess.check_output(
    ["databricks", "auth", "token", "--host", host],
    text=True
).strip()
os.environ["DATABRICKS_TOKEN"] = token
print(f"Token obtained for {host}")

In [ ]:
# Push all models
!cd /workspaces/your-project/local-devex/dbt && dbt run --target prod

In [ ]:
# Push a single model
!cd /workspaces/your-project/local-devex/dbt && dbt run --target prod --select events_enriched